# Time series for newcomers: what generated series are for

Deep-RL papers often open with a tiny gridworld: far too simple to be useful, but it shows in one picture what the work is about. This notebook does the same for the time-series transformations of `kcai-data-sampling`, on the gridworld of time series: **Air Passengers**, twelve years of monthly airline passengers (Box & Jenkins, 1976), known from every forecasting textbook.

The story: an airline planner looks at the record and asks two questions.

1. **Forecasting**: how many passengers next year, month by month?
2. **Anomaly detection**: is this year normal, or did something happen?

The record is clean: twelve years, **no anomaly**. The transformations create the situations that are not in it. At the end, the **Nile** shows what a real anomaly looks like.

**How this notebook works.** Everything is computed here, step by step, with the package's own objects:

1. the **loader** cuts the series into windows, the samples;
2. the **transformation classes** turn a window into a new window, and say what they did;
3. a list of transformations, with swept parameters, turns the record into **test sets**;
4. two tiny models are evaluated on them;
5. at the end, the same test sets are produced by the **pipeline** from the YAML configs of `examples/config/`, which is how the package is used outside a notebook, and we check that they are the same windows.

**Data.** Air Passengers and Nile, from the R `datasets` package through [Rdatasets](https://github.com/vincentarelbundock/Rdatasets), downloaded by `scripts/fetch_air_passengers_nile.py` (the first cell runs it if needed). Nile is public domain; the licence of Air Passengers is not established for certain (Rdatasets believes it free to redistribute), so the outputs of this notebook are kept out of the repository.

**Models.** The simplest forecasts there are, in `air_passengers_models.py` next to this notebook. This is a toy, and it says so.

```bash
uv run --with matplotlib --with ipykernel jupyter lab
```

The figures are those of the slide deck (*the gridworld of time series*), saved as PNG in `examples/outputs/air-passengers-demo/figures/`. Seed 0 everywhere.

## Setup

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyBboxPatch, Polygon, Rectangle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import yaml

%config InlineBackend.figure_format = 'png'
plt.rcParams["font.size"] = 15  # the only change from matplotlib's defaults: readable on a slide
pd.set_option("display.max_colwidth", None)

# Work from the repository root: every path in the configs is relative to it.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
os.chdir(REPO)
sys.path.insert(0, str(REPO / "examples" / "notebooks"))

# The package's objects: the loader that cuts windows, the transformations, the sweeps, the pipeline.
from kcai_data_sampling_core.models.sweep import SweepConfig
from kcai_data_sampling_job.cli import run
from kcai_data_sampling_job.dataloaders.api.timeseries import TimeSeriesLoaderConfig, TimeSeriesParquetDataLoader
from kcai_data_sampling_timeseries.api.transformations import (
    AddSinusoid,
    ColoredNoise,
    Drift,
    GarchNoise,
    ImpulseNoise,
    Jitter,
    LevelShift,
    LowPass,
    OUNoise,
)

import air_passengers_models as am

DATA = Path("examples/data/air_passengers_nile")
if not (DATA / "air_passengers.parquet").exists():
    subprocess.run([sys.executable, "scripts/fetch_air_passengers_nile.py"], check=True)

FIGURES = Path("examples/outputs/air-passengers-demo/figures")  # PNG copies for the slides (git-ignored)
FIGURES.mkdir(parents=True, exist_ok=True)


def save(fig, name):
    # Each figure has the size of its slot in the deck: no tight bounding box, the aspect is kept.
    fig.savefig(FIGURES / f"{name}.png", dpi=200)


def years_axis(ax, step=1):
    ax.xaxis.set_major_locator(mdates.YearLocator(step))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

## 1. A time series in one picture

The data file is a parquet table with one row per series. Air Passengers is a single series of 144 months; the fetch script stores the counts (`values`, thousands of passengers) and their logarithm (`log_values`), each as a `(T, 1)` array: T steps, one channel.

In [ ]:
table = pq.read_table(DATA / "air_passengers.parquet").to_pandas()
LOG = am.load_series(DATA / "air_passengers.parquet")  # log(thousands of passengers), (144,)
PASSENGERS = np.exp(LOG)
MONTHS = pd.date_range(table.start[0], periods=len(LOG), freq="MS")
print(f"{len(table)} series, {len(LOG)} steps from {table.start[0]}, frequency {table.freq[0]}")
table.assign(**{c: table[c].map(lambda v: f"({len(v)}, 1): {np.round(np.concatenate(v[:3]), 2).tolist()} ...") for c in ("values", "log_values")})

Only two things to see: it **grows**, and it **repeats every year** (a summer peak).

The yearly swing grows with the traffic: 44 thousand passengers in 1949, 232 thousand in 1960. So most of this notebook describes changes **in percent**, which means working on the logarithm of the series: "10 % fewer passengers" means the same thing in 1949 and in 1960, "30 thousand fewer" does not. On the log scale an offset of -0.19 is about -17 %, a slope of -0.02 about -2 % per month, a noise of 0.02 about 2 %.

In [ ]:
fig, ax = plt.subplots(figsize=(12.2, 6.2))
ax.plot(MONTHS, PASSENGERS, label="monthly passengers")
ax.plot(pd.date_range("1949-07", periods=12, freq="12MS"), PASSENGERS.reshape(-1, 12).mean(axis=1), marker="o", label="yearly mean: it grows")
ax.annotate("summer peak, every year", (MONTHS[127], PASSENGERS[127]), xytext=(MONTHS[62], 575), fontsize=14,
            arrowprops={"arrowstyle": "-", "color": "gray", "lw": 1})
ax.set_ylabel("thousands of passengers")
ax.legend(loc="upper left")
fig.tight_layout()
save(fig, "fig_series")
plt.show()

## 2. Samples are cut, not given

In image classification a sample is given: one image, collected as such, with an annotation outside the pixels. A time series is one long record; **the samples are windows cut out of it**, and the cut is a choice.

The loader makes that choice explicit. Its configuration is the same block as in the YAML configs (`dataloaders.loaders`): which file, which column holds the values, and how to cut. Here, windows of **36 months** (24 months of history, 12 to test), **one starting every month**.

In [ ]:
LOADER = {
    "name": "air_passengers",
    "type": "timeseries_parquet",
    "path": "examples/data/air_passengers_nile/air_passengers.parquet",
    "id_column": "series_id",
    "values_column": "log_values",  # the log scale: parameters read in percent
    "load_batch_size": 128,
    "windows": {"mode": "consecutive", "window": 36, "stride": 1},
}


def select(loader):
    """Cut the series into windows: the loader's selection, as a list of batches."""
    config = TimeSeriesLoaderConfig.model_validate(loader)
    (selection,) = TimeSeriesParquetDataLoader(config.name, config).get_selections()
    selection.bootstrap(None)
    return list(selection)


def keys_of(batches):
    """Where every window comes from: one row per window, with its id."""
    keys = pd.concat([b.columns.to_pandas().assign(id=b.ids) for b in batches], ignore_index=True)
    return keys[["series_id", "start_window", "end_window", "id"]]


def window_at(batches, start):
    """The one-window batch that starts at step ``start``."""
    for batch in batches:
        hit = np.flatnonzero(batch.columns.to_pandas().start_window == start)
        if len(hit):
            return batch.row(int(hit[0]))
    raise KeyError(start)


BATCHES = select(LOADER)
KEYS = keys_of(BATCHES)
print(f"{len(KEYS)} windows in {len(BATCHES)} batch(es); a batch holds a (windows, steps, channels) array: {BATCHES[0].data.shape}")
KEYS.head()

144 months give 109 windows of 36 months. They **overlap** (two neighbours share 35 months) and they are **ordered in time**, so a train / test split is made in time, never at random. Each window has an id, derived from where it was cut: every sample generated from it will point back to it.

We follow one window through the notebook: the one starting in **January 1955** (step 72), covering 1955-1957.

In [ ]:
START = 72  # January 1955
WINDOW = window_at(BATCHES, START)
months = MONTHS[START : START + 36]
print(WINDOW.columns.to_pandas()[["series_id", "start_window", "end_window"]].to_dict("records")[0], "| id", WINDOW.ids[0])

**One transformation, one new sample.** A transformation class is built from its parameters, then `transform` takes a batch of windows and returns one `Output` per window: the new window `x`, and everything needed to say what was done to it (`algorithm`, `params`, `seed`, and `parent_id`, the window it comes from).

Here, `level_shift`: from 3/4 of the window on (month 27 of 36), the series is moved down by 0.19 on the log scale, about **-17 %**: a demand drop that lasts.

In [ ]:
drop = LevelShift({"offset": -0.19, "start": 0.75})
(out,) = drop.transform(WINDOW)
print("new window:", out.x.shape)
{k: v for k, v in out.row().items() if k in ("id", "parent_id", "algorithm", "params", "seed")}

What happens to the label is a choice of the use case, not of the transformation:

- **forecasting**: the label is the future of the same series, inside the signal. To keep it, we transform the history only and read the true future from the record;
- **anomaly detection**: the record has no label at all. A transformation can stay "normal" (harmless variability) or **create** the label "abnormal" (a fault, with its type, size and position, all in `params`).

In [ ]:
def image_panel(ax):
    """Image classification: a picture and its annotation (a drawing, for the comparison)."""
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 6)
    ax.axis("off")
    ax.add_patch(Rectangle((0.6, 0.6), 5.2, 4.6, facecolor="none", edgecolor="black", lw=1.5))
    ax.add_patch(Circle((4.4, 3.9), 0.55, color="C1"))
    ax.add_patch(Polygon([(0.6, 0.6), (2.4, 3.3), (3.5, 2.0), (4.4, 2.8), (5.8, 0.6)], closed=True, color="C0"))
    ax.add_patch(FancyBboxPatch((6.5, 2.4), 3.1, 1.0, boxstyle="round,pad=0.12", facecolor="white", edgecolor="black", lw=1.2))
    ax.text(8.05, 2.9, 'label: "mountain"', ha="center", va="center", fontsize=13)
    ax.plot([5.85, 6.4], [2.9, 2.9], color="black", lw=1.2)
    ax.set_title("One image, one annotation", loc="left")


fig, axes = plt.subplots(1, 3, figsize=(18.1, 3.3))
image_panel(axes[0])
ax = axes[1]
ax.plot(MONTHS, PASSENGERS, color="gray", lw=1, alpha=0.6)
ax.plot(MONTHS[START : START + 24], PASSENGERS[START : START + 24], label="history")
ax.plot(MONTHS[START + 24 : START + 36], PASSENGERS[START + 24 : START + 36], lw=2.5, label="future = label")
ax.set_ylim(80, 820)
years_axis(ax, 2)
ax.legend(loc="upper left", ncol=2, fontsize=12)
ax.set_title("A window cut out of the series", loc="left")
ax = axes[2]
ax.plot(months, PASSENGERS[START : START + 36], label="recorded")
ax.plot(months, np.exp(out.x[:, 0]), ls="--", label="generated: label abnormal,\nlevel_shift -17 % from month 27")
ax.set_ylim(200, 640)
years_axis(ax)
ax.legend(loc="upper left", fontsize=12)
ax.set_title("The same window, with a generated fault", loc="left")
fig.tight_layout(w_pad=2.5)
save(fig, "fig_samples")
plt.show()

## 3. A transformation is a situation

Each transformation stands for something that could happen to the airline, described with a few parameters in percent:

| Transformation | Situation | Role in the detection test |
|---|---|---|
| `jitter`, `sigma` 0.02 | counting noise, about 2 % | normal: must **not** raise an alert |
| `level_shift`, `offset` -0.19 from 3/4 of the window | demand drops at once and stays down (a lasting strike, a new competitor), about -17 % | fault: **must** raise an alert |
| `drift`, `slope` -0.02 | demand slowly collapses, about -2 % per month | fault |
| `add_sinusoid`, one cycle per year, `amplitude` 0.2 | the season is reshaped, about ±20 % | fault |

`jitter` draws random numbers, so it needs a **seed**: each window draws from its own generator, seeded by the seed, the window's id and the parameters, so the same window always gets the same noise, whatever the batch it is in. The other three are deterministic.

In [ ]:
SITUATIONS = {
    "Counting noise": Jitter({"sigma": 0.02, "seed": 0}),
    "Demand drop": LevelShift({"offset": -0.19, "start": 0.75}),
    "Slow collapse": Drift({"slope": -0.02}),
    "Season reshaped": AddSinusoid({"frequency": 0.0833333, "amplitude": 0.2, "phase": 1.5707963}),  # 1/12 cycle per month, peak in summer
}
SITUATION_OUTPUTS = {title: t.transform(WINDOW)[0] for title, t in SITUATIONS.items()}

fig, axes = plt.subplots(2, 2, figsize=(12.0, 6.5), sharex=True, sharey=True)
for ax, (title, output) in zip(axes.flat, SITUATION_OUTPUTS.items()):
    ax.plot(months, PASSENGERS[START : START + 36], label="recorded")
    ax.plot(months, np.exp(output.x[:, 0]), label="generated")  # back from the log scale to thousands
    ax.set_title(title, loc="left")
    years_axis(ax)
for ax in axes[:, 0]:
    ax.set_ylabel("thousands")
axes[0, 0].legend(loc="upper left", ncol=2)
fig.tight_layout()
save(fig, "fig_situations")
plt.show()

## 4. Time-series noises

`jitter` is white noise: every month independent of the others. Real disturbances have a memory. Four noises take time into account, and each one is either **additive** (`x + n`: the same size whatever the level) or **multiplicative** (`x · (1 + n)`: proportional to the level), chosen with `mode`.

For that choice to keep its meaning, these run on the **raw counts**, not on the log scale: a second selection, same windows, `values_column: values`.

| Transformation | Situation | Parameters |
|---|---|---|
| `colored_noise` | reporting drift: pink (1/f) noise, slow wanderings dominate | `sigma` 10 thousand, `exponent` 1, additive |
| `ou_noise` | fluctuating offset that comes back (Ornstein-Uhlenbeck), correlated over about 5 months | `sigma` 15 thousand, `theta` 0.2, additive |
| `garch_noise` | calm and turbulent periods (GARCH volatility), bursts that grow with the traffic | `sigma` 5 %, `alpha` 0.15, `beta` 0.8, multiplicative |
| `impulse_noise` | counting glitches: a few miscounted months | `rate` 5 % of the months, `amplitude` ±60 thousand, additive |

In [ ]:
RAW = select({**LOADER, "values_column": "values"})  # thousands of passengers
RAW_WINDOW = window_at(RAW, START)

NOISES = {
    "Pink noise (1/f), additive": ColoredNoise({"sigma": 10.0, "exponent": 1.0, "seed": 0}),
    "Ornstein-Uhlenbeck, additive": OUNoise({"sigma": 15.0, "theta": 0.2, "seed": 0}),
    "GARCH, multiplicative": GarchNoise({"sigma": 0.05, "alpha": 0.15, "beta": 0.8, "mode": "multiplicative", "seed": 0}),
    "Impulse spikes, additive": ImpulseNoise({"rate": 0.05, "amplitude": 60.0, "seed": 0}),
}
NOISE_OUTPUTS = {title: t.transform(RAW_WINDOW)[0] for title, t in NOISES.items()}

fig, axes = plt.subplots(2, 2, figsize=(12.0, 6.5), sharex=True, sharey=True)
for ax, (title, output) in zip(axes.flat, NOISE_OUTPUTS.items()):
    ax.plot(months, RAW_WINDOW.data[0, :, 0], label="recorded")
    ax.plot(months, output.x[:, 0], label="generated")
    ax.set_title(title, loc="left")
    years_axis(ax)
for ax in axes[:, 0]:
    ax.set_ylabel("thousands")
axes[0, 0].legend(loc="upper left", ncol=2)
fig.tight_layout()
save(fig, "fig_noises")
plt.show()

## 5. Generated data is labelled data

Every output carries its own record: where it comes from and what was done to it. Written by the pipeline, these records are the rows of the **ledger**. Here are the eight windows generated above from the 1955-1957 window. The record has no labelled anomaly: we did not need one, the ledger is the label (which fault, of what size, in which window).

In [ ]:
pd.DataFrame([
    {"source window": f"months {START}–{START + 35}", **{k: o.row()[k] for k in ("algorithm", "params", "seed", "parent_id", "id")}}
    for o in [*SITUATION_OUTPUTS.values(), *NOISE_OUTPUTS.values()]
])

## 6. From one window to test sets: sweeping parameters

One window and one parameter value tell a story; a test needs every window, and **a range of sizes** for each fault. A sweep is written as a `SweepConfig`, the same block as in the YAML configs: an interval and a number of values. It expands into one transformation per value.

Then `generate` applies every transformation to every window and keeps one record per output: a small ledger, in memory.

In [ ]:
def sweep(make, **sweep_config):
    """One transformation per value of a sweep; ``make`` builds it from the value."""
    return [make(v) for v in SweepConfig(**sweep_config).values()]


def generate(batches, transformations):
    """Apply every transformation to every window: the records (a ledger) and the new windows."""
    rows, windows = [], []
    for batch in batches:
        keys = batch.columns.to_pandas()[["series_id", "start_window", "end_window"]].to_dict("records")
        for t in transformations:
            for key, o in zip(keys, t.transform(batch), strict=True):
                rows.append({**key, "algorithm": o.algorithm, "params": o.params, "seed": o.seed, "parent_id": o.parent_id, "id": o.id})
                windows.append(o.x[:, 0])
    ledger = pd.DataFrame(rows)
    ledger["value"] = [p[SWEPT[a]] for a, p in zip(ledger.algorithm, ledger.params)]  # the x axis of the curves
    return ledger, np.stack(windows)


#: The parameter each curve is drawn against.
SWEPT = {"jitter": "sigma", "low_pass": "cutoff", "level_shift": "offset", "drift": "slope", "add_sinusoid": "amplitude",
         "colored_noise": "sigma", "ou_noise": "sigma", "garch_noise": "sigma", "impulse_noise": "amplitude"}


# Anomaly detection: one normal variability, three faults of six sizes each, on the 109 windows of 36 months.
ANOMALY_TRANSFORMATIONS = [
    Jitter({"sigma": 0.02, "seed": 0}),
    *sweep(lambda v: LevelShift({"offset": v, "start": 0.75}), range=[-0.4, -0.05], samples=6, round=4),
    *sweep(lambda v: Drift({"slope": v}), range=[-0.02, -0.002], samples=6, round=4),
    *sweep(lambda v: AddSinusoid({"frequency": 0.0833333, "phase": 1.5707963, "amplitude": v}), range=[0.02, 0.2], samples=6, round=4),
]
ANOMALY, ANOMALY_WINDOWS = generate(BATCHES, ANOMALY_TRANSFORMATIONS)
print(f"{len(ANOMALY_TRANSFORMATIONS)} transformations x {len(KEYS)} windows = {len(ANOMALY)} generated windows")
ANOMALY.groupby("algorithm", sort=False).value.unique().to_frame("swept values")

`round: 4` keeps four decimals on the swept values (the default is 2, too coarse for slopes of a few thousandths).

Forecasting needs other windows: **24 months of history**, the future being read from the record. Two degradations of the history, swept: counting noise from 0 to 10 %, and smoothed reporting (`low_pass`, the higher the cutoff, the less smoothing).

In [ ]:
FORECAST_LOADER = {**LOADER, "windows": {"mode": "consecutive", "window": 24, "stride": 1}}
FORECAST_BATCHES = select(FORECAST_LOADER)
FORECAST_TRANSFORMATIONS = [
    *sweep(lambda v: Jitter({"sigma": v, "seed": 0}), range=[0.0, 0.1], samples=5, round=4),
    *sweep(lambda v: LowPass({"cutoff": v}), range=[0.1, 0.3], samples=3, round=4),
]
FORECAST_ALL, FORECAST_ALL_WINDOWS = generate(FORECAST_BATCHES, FORECAST_TRANSFORMATIONS)
print(f"{len(FORECAST_TRANSFORMATIONS)} transformations x {len(keys_of(FORECAST_BATCHES))} windows = {len(FORECAST_ALL)} generated histories")

## 7. Forecasting next year

Two forecasts a newcomer can do by hand:

- **this year**: next year is this year (seasonal naive);
- **this year + growth**: next year is this year, plus the growth between the last two years.

Each generated window is a degraded history; the **true next year is read from the recorded series** after the window's end, so it is never transformed. On the log scale, the error reads as a percentage. Histories whose next year runs past 1960 are dropped.

In [ ]:
keep = (FORECAST_ALL.end_window + 12 < len(LOG)).to_numpy()
FORECAST, FORECAST_WINDOWS = FORECAST_ALL[keep].reset_index(drop=True), FORECAST_ALL_WINDOWS[keep]
TRUTH = np.stack([LOG[e + 1 : e + 13] for e in FORECAST.end_window])
MODELS = {"this year + growth": am.seasonal_growth, "this year": am.seasonal_naive}
for name, model in MODELS.items():
    FORECAST[name] = 100 * np.abs(model(FORECAST_WINDOWS) - TRUTH).mean(axis=1)
ERRORS = FORECAST.groupby(["algorithm", "value"])[list(MODELS)].mean()
ERRORS.round(1)

In [ ]:
END = 107  # history up to December 1957, forecast of 1958
clean = FORECAST.index[(FORECAST.end_window == END) & (FORECAST.algorithm == "jitter") & np.isclose(FORECAST.value, 0.0)][0]
noisy = FORECAST.index[(FORECAST.end_window == END) & (FORECAST.algorithm == "jitter") & np.isclose(FORECAST.value, 0.1)][0]

fig, axes = plt.subplots(1, 2, figsize=(12.4, 5.2), gridspec_kw={"width_ratios": [1.15, 1]})
ax = axes[0]
ax.plot(MONTHS[END - 23 : END + 1], PASSENGERS[END - 23 : END + 1], label="history")
ax.plot(MONTHS[END + 1 : END + 13], PASSENGERS[END + 1 : END + 13], lw=3, label="true 1958")
ax.plot(MONTHS[END + 1 : END + 13], np.exp(am.seasonal_growth(FORECAST_WINDOWS[[clean]])[0]), ls="--", label="forecast, clean history")
ax.plot(MONTHS[END + 1 : END + 13], np.exp(am.seasonal_growth(FORECAST_WINDOWS[[noisy]])[0]), ls=":", label="forecast, 10 % noise")
ax.set_title("Forecasting 1958", loc="left")
ax.set_ylabel("thousands")
years_axis(ax)
ax.legend(loc="upper left", fontsize=12)
ax = axes[1]
curve = ERRORS.loc["jitter"]
for name in MODELS:
    ax.plot(curve.index * 100, curve[name], marker="o", label=name)
ax.set_title("Error vs. noise in the history", loc="left")
ax.set_xlabel("noise (about %)")
ax.set_ylabel("error on next year (%)")
ax.set_ylim(0, 16)
ax.legend(loc="lower right", fontsize=12)
fig.tight_layout()
save(fig, "fig_forecasting")
plt.show()
print(f"this year + growth: {ERRORS.loc[('jitter', 0.0), 'this year + growth']:.1f} % on a clean history, "
      f"{ERRORS.loc[('jitter', 0.1), 'this year + growth']:.1f} % with 10 % noise")

Adding the growth halves the error of the plain seasonal forecast, and the noisier the history, the worse both get: the generated histories turn "how robust is my forecast?" into a curve.

## 8. No anomaly in the record? We make them

A toy detector: forecast the last year of each 36-month window from the two years before (this year + growth), and raise an alert when the error is larger than on 95 % of the **recorded** windows, the 109 windows of the selection. So about 5 % of normal windows raise an alert, by construction.

The label comes from the ledger: we know which fault was injected, of what size, into which window.

In [ ]:
RECORDED = np.concatenate([b.data[:, :, 0] for b in BATCHES])  # the 109 recorded windows, (109, 36)
DETECTOR = am.Detector().fit(RECORDED)
ANOMALY["alert"] = DETECTOR.alerts(ANOMALY_WINDOWS)
FALSE_ALARMS = DETECTOR.alerts(RECORDED).mean() * 100
RATES = ANOMALY.groupby(["algorithm", "value"]).alert.mean() * 100

curve = RATES.loc["level_shift"].sort_index(ascending=False)  # smallest drop first
drops = pd.Series(curve.values, index=np.round(100 * (1 - np.exp(curve.index.to_numpy()))).astype(int))
fig, ax = plt.subplots(figsize=(12.4, 5.9))
ax.plot(drops.index, drops.values, marker="o", label="sudden demand drop (level_shift)")
ax.axhline(FALSE_ALARMS, color="gray", lw=1.2, ls="--")
ax.text(drops.index.max(), FALSE_ALARMS + 2.5, f"slow collapse, reshaped season, recorded windows: {FALSE_ALARMS:.1f} %",
        ha="right", va="bottom", color="gray", fontsize=13)
for x, y in drops.items():
    ax.annotate(f"{y:.0f} %", (x, y), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=13)
ax.set_xlabel("size of the drop (%)")
ax.set_ylabel("windows raising an alert (%)")
ax.set_ylim(0, 112)
ax.set_xticks(drops.index)
ax.legend(loc="center right")
fig.tight_layout()
save(fig, "fig_detection")
plt.show()

pd.DataFrame({
    "situation": ["recorded windows", "counting noise 2 % (jitter, normal)", "demand drop -17 % (level_shift)",
                  "slow collapse, any size (drift)", "season reshaped, any size (add_sinusoid)"],
    "windows raising an alert (%)": [FALSE_ALARMS, RATES.loc[("jitter", 0.02)], RATES.loc[("level_shift", -0.19)],
                                     RATES.loc["drift"].max(), RATES.loc["add_sinusoid"].max()],
}).round(1)

- A **sudden drop** is caught once it is large enough: we read off the smallest drop the detector sees.
- A **slow collapse** and a **reshaped season** are not caught at all, whatever their size: their alert rate stays exactly at the false-alarm level. The detector copies last year's shape, so a reshaped season is simply forecast; and it estimates the growth from the history, so a steady collapse is absorbed as "the new trend". These faults are invisible to it by construction, and the generated faults reveal this blind spot before a real event does.

## 9. What a real anomaly looks like

The Nile at Aswan, 1871-1970: around 1899 its yearly flow drops at once and stays lower (the first Aswan dam was built in those years). That is a **level shift**, the kind of break `level_shift` generates: next to it, the -17 % drop generated in section 2.

In [ ]:
NILE = am.load_series(DATA / "nile.parquet", column="values")
years = np.arange(1871, 1871 + len(NILE))
before, after = NILE[years <= 1898].mean(), NILE[years >= 1899].mean()

fig, axes = plt.subplots(1, 2, figsize=(12.4, 5.9))
ax = axes[0]
ax.plot(years, NILE, label="yearly flow")
ax.plot([1871, 1898], [before, before], color="C1", lw=3, label="mean before / after 1899")
ax.plot([1899, 1970], [after, after], color="C1", lw=3)
ax.set_title("A real break: the Nile", loc="left")
ax.set_ylabel("yearly flow (10⁸ m³)")
ax.legend(loc="upper right", fontsize=12)
ax = axes[1]
ax.plot(months, PASSENGERS[START : START + 36], label="recorded")
ax.plot(months, np.exp(out.x[:, 0]), ls="--", label="generated: -17 %")
ax.set_title("A generated break", loc="left")
ax.set_ylabel("thousands of passengers")
years_axis(ax)
ax.legend(loc="upper left", fontsize=12)
fig.tight_layout()
save(fig, "fig_nile")
plt.show()
print(f"Nile: mean {before:,.0f} before 1899, {after:,.0f} after ({100 * (after / before - 1):+.0f} %)")

## 10. The same test sets, from a config

Outside a notebook, nobody writes these loops: a YAML config names the loader and the transformations, and the pipeline (`kcai-data-sampling process -p <config>`, or `run(config)` from Python) writes the ledger as a parquet table and one `.npy` file per generated window. Here is the anomaly config; its `dataloaders` and `transformations` blocks are the ones built by hand above.

In [ ]:
print(Path("examples/config/air-passengers-anomaly.yaml").read_text())

We run the three configs and check that the pipeline wrote **the same windows** as this notebook: same ids, same values. The ids match because they are derived from the same things (the source window, the algorithm, its parameters and seed), and the values match because each window's noise is seeded by its own id.

In [ ]:
NOISES_ALL, NOISES_ALL_WINDOWS = generate(RAW, list(NOISES.values()))
IN_NOTEBOOK = {
    "air-passengers-anomaly": (ANOMALY, ANOMALY_WINDOWS),
    "air-passengers-forecasting": (FORECAST_ALL, FORECAST_ALL_WINDOWS),
    "air-passengers-noises": (NOISES_ALL, NOISES_ALL_WINDOWS),
}
for name, (ledger, windows) in IN_NOTEBOOK.items():
    with open(f"examples/config/{name}.yaml") as f:
        config = yaml.safe_load(f)
    config["compute"] = {**config.get("compute", {}), "progress_bar": False}  # quiet in a notebook
    shutil.rmtree(f"examples/outputs/{name}", ignore_errors=True)
    run(config)
    root = Path(f"examples/outputs/{name}/air_passengers")
    written = pq.read_table(root / "ledger.parquet").to_pandas()
    mine = dict(zip(ledger.id, windows))
    same = set(written.id) == set(mine) and all(
        np.allclose(np.load(root / "payloads" / a)[:, 0], mine[i]) for i, a in zip(written.id, written.artifact)
    )
    print(f"{name}: {len(written)} rows in the ledger, the same windows as in the notebook: {same}")

written.head(3)  # the ledger of the noises run: one row per generated window

## 11. What this shows

- **A sample is a window**, cut by the loader; a transformation turns it into a new window and keeps a record of what it did.
- **A transformation is a situation**, described with a few physical parameters ("17 % fewer passengers from month 27").
- **Generated data is labelled data**: the record says what was done to each window, so a record with no anomaly becomes a test set with known anomalies.
- **Sweeping a parameter gives a curve**: how fast a forecast degrades, the smallest fault a detector sees, and what it cannot see at all.

The same steps run on real data in `ett-demo.ipynb` (oil temperature of power transformers): the gridworld first, then the real environment.